# Artificial Intelligence — Module 01
## Notebook 02: Knowledge Representation

Course by **Prof. Dr. Érica da Costa Reis Carvalho** — UFSJ  
Notebook by: **Caio Fromm** (Repository Author)

### Learning Objectives

By the end of this notebook, you should be able to:

- Explain the purpose and properties of Knowledge Representation (KR).
- Understand and build simple **Semantic Networks**.
- Understand and structure knowledge using **Frames**.
- Understand and outline event sequences using **Scripts**.

## 1. Why Knowledge Representation?

Knowledge Representation (KR) is a central part of Artificial Intelligence. It's about how we can take real-world facts, relationships, and "common sense" and store it in a way a computer can understand and reason with.

The goal is to reduce complex intelligent actions (like diagnosis, planning, or conversation) into a problem of **search** and **inference** over a structured knowledge base.

A good representation (based on *Aula 03*) should be:

- **Transparent:** We can understand what is being said.
- **Fast:** We can store and retrieve information quickly.
- **Computable:** We can create and manipulate it with an algorithm.

## 2. Semantic Networks

A semantic network is one of the simplest and most intuitive KR methods. It's a graph where:

- **Nodes** represent objects or concepts.
- **Links (Edges)** represent relationships between those concepts.

These links are labeled to show the *type* of relationship.

*(A diagram of a semantic network would show nodes like 'Animal', 'Mammal', and 'Dog' connected by 'is_a' or 'a_kind_of' links.)*

Common relationships (based on *Aula 03, slide 15*):

- **`é_um` (is_a):** Relates an instance to its class (e.g., "Rex" *is_a* "Cão").
- **`Ako` (a_kind_of):** Relates a class to a superclass (e.g., "Cão" *a_kind_of* "Mamífero"). This allows **inheritance**.
- **`tem_um` (has_a) / `parte_de` (part_of):** Identifies attributes or parts (e.g., "Mamífero" *has_a* "Pêlos").

One powerful feature is **transitivity**. If `Cão -> Ako -> Mamífero` and `Mamífero -> Ako -> Animal`, we can infer that `Cão -> Ako -> Animal`.

### Code Example: Building a Semantic Network

We can use the `networkx` library to build and visualize the "Animal" network from *Aula 03 (slides 11-13)*.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1. Create a directed graph
G = nx.DiGraph()

# 2. Add nodes (concepts)
nodes = ["Animal", "Comer", "Mamífero", "Pássaro", "Pêlos", "Cão"]
G.add_nodes_from(nodes)

# 3. Add edges (relationships)
G.add_edge("Animal", "Comer", label="faz")
G.add_edge("Mamífero", "Animal", label="é_um")
G.add_edge("Pássaro", "Animal", label="é_um")
G.add_edge("Mamífero", "Pêlos", label="tem")
G.add_edge("Cão", "Mamífero", label="é_um")

# 4. Draw the network
try:
    # 'spring_layout' gives a nice automatic layout
    pos = nx.spring_layout(G, seed=42, k=1.5)
    
    plt.figure(figsize=(8, 6))
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_size=2500, node_color="lightblue", alpha=0.8)
    
    # Draw labels (nodes)
    nx.draw_networkx_labels(G, pos, font_size=10, font_weight="bold")
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, node_size=2500, arrowstyle="->", arrowsize=20, edge_color="gray")
    
    # Draw edge labels
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
    
    plt.title("Semantic Network Example (Aula 03)")
    plt.axis('off') # Hide the axes
    plt.show()

except Exception as e:
    print("Could not draw plot. Make sure matplotlib and networkx are installed.")
    print("Error:", e)
    print("\nNetwork Edges:", G.edges(data=True))

### Exercise: Build Your Own Network

*(Based on Aula 03, slide 30)*

**Text:** "O Rio de Janeiro é uma cidade localizada no Brasil. Ela é famosa pela sua beleza natural e pontos turísticos como o Cristo Redentor, o Pão de Açúcar e suas praias. O Cristo Redentor, uma das sete maravilhas do mundo moderno, fica no topo do Corcovado, montanha que é um símbolo da cidade."

**Task:** Identify the objects (nodes) and relationships (edges) from the text. Add them to a new `networkx` graph and try to visualize it.

In [ ]:
# Solution space for the exercise
G_Rio = nx.DiGraph()

# Example relationships (from Aula 04, slide 3):
G_Rio.add_edge("Rio de Janeiro", "Cidade", label="é_um")
G_Rio.add_edge("Rio de Janeiro", "Brasil", label="localizada_em")
G_Rio.add_edge("Cristo Redentor", "Rio de Janeiro", label="fica_em")
G_Rio.add_edge("Pão de Açúcar", "Rio de Janeiro", label="fica_em")
G_Rio.add_edge("Cristo Redentor", "Corcovado", label="topo_de")
G_Rio.add_edge("Rio de Janeiro", "Beleza Natural", label="famosa_por")

print("Rio Network Edges:", G_Rio.edges(data=True))

## 3. Frames

Frames are a more structured way to represent knowledge, very similar to an "object" in programming. They are templates for concepts that have **slots** (attributes) and **values**.

*(A diagram of a frame would show a box labeled 'Dog' with slots like 'is_a: Mammal', 'color: default brown', 'legs: 4'.)*

Key features of Frames:

- **Slots:** Attributes of the concept (e.g., `Nome`, `Raça`, `Pêlo`).
- **Values:** The specific data in a slot. This can be:
  - A fixed value (e.g., `Nome: "Rex"`)
  - A **default** value (e.g., `Raça: "Mongrel"` (Vira-lata))
  - A range or type (e.g., `Sexo: "Macho ou Fêmea"`)
- **Inheritance:** A frame can be an instance of a class (e.g., "Rex" *is_a* "Cão") or a subclass of another (e.g., "Cão" *Ako* "Mamífero"). It inherits all the slots from its parent.
- **Demons:** Procedures attached to slots. These are like triggers. For example, a `IF_NEEDED` demon on an `Area` slot could *calculate* the area from `Width` and `Height` slots.

### Code Example: Frames as Dictionaries

We can easily represent frames and instances as Python dictionaries.

In [ ]:
# Class Frame for "Cão" (based on Aula 03, slide 21)
frame_cao = {
    "name": "Cão",
    "Ako": "Mamífero", # Inheritance
    "slots": {
        "Nome": {"type": "string"}, 
        "Raça": {"default": "Mongrel"}, # Vira-lata
        "Pêlo": {"default": "Longo"},
        "Sexo": {"range": ["Macho", "Fêmea"]}
    }
}

# Instance Frame for "Rex" (based on Aula 03, slide 23)
instance_rex = {
    "is_a": "Cão",
    "slots": {
        "Nome": "Rex",
        "Raça": "German Shepherd",
        "Pêlo": "Longo",
        "Sexo": "Macho"
    }
}

print("--- Generic Frame: Cão ---")
print(frame_cao)
print("\n--- Instance Frame: Rex ---")
print(instance_rex)

### Exercise: Model Rules as Frames

*(Based on Aula 03, slide 31)*

**Task:** Model the following rules using a Frame structure. How would you represent the `Carro` (Car) class and its instances?

```
OBJETIVO = carro

SE tamanho_família = muitas pessoas
ENTÃO tamanho_carro = grande

SE uso_familia = passeio
E tamanho_carro = pequeno
E disponibilidade de dinheiro = média
ENTÃO carro = Corsa

SE opção = passeio
E tamanho_carro = grande
E disponibilidade de dinheiro = grande
ENTÃO carro = Captiva
```

In [ ]:
# Solution space for the exercise

# We could define a generic frame for 'Carro'
frame_carro = {
    "name": "Carro",
    "slots": {
        "tamanho_carro": {"range": ["pequeno", "médio", "grande"]},
        "uso": {"range": ["passeio", "trabalho"]},
        "dinheiro": {"range": ["baixa", "média", "grande"]}
    }
}

# And the rules could define specific instances or subclasses
frame_corsa = {
    "is_a": "Carro",
    "name": "Corsa",
    "slots": {
        "tamanho_carro": "pequeno",
        "uso": "passeio",
        "dinheiro": "média"
    }
}

frame_captiva = {
    "is_a": "Carro",
    "name": "Captiva",
    "slots": {
        "tamanho_carro": "grande",
        "uso": "passeio",
        "dinheiro": "grande"
    }
}

print(frame_corsa)
print(frame_captiva)

## 4. Scripts

While Frames represent *objects*, **Scripts** represent stereotypical *event sequences*. Think of them as a script for a play or movie.

They are designed to help an AI understand and predict sequences of actions in a common situation (e.g., "going to a restaurant", "visiting a doctor").

A Script has key elements (based on *Aula 03, slide 33*):

- **Papéis (Roles):** The people involved (e.g., `Freguês` (Customer), `Garçom` (Waiter)).
- **Objetos (Objects):** Relevant items (e.g., `Mesas`, `Menu`, `Dinheiro`).
- **Condições de Entrada (Entry Conditions):** What must be true for the script to start (e.g., `Freguês está com fome`, `Freguês tem dinheiro`).
- **Cenas (Scenes):** The sequence of events (e.g., `Cena 1: Entrar`, `Cena 2: Pedir`).
- **Resultados (Outcomes):** What is true after the script finishes (e.g., `Freguês não tem fome`, `Freguês tem menos dinheiro`).

### Code Example: Restaurant Script as a Dictionary

In [ ]:
# A simple 'Restaurant' script (based on Aula 03, slides 36-37)
restaurant_script = {
    "name": "Script Restaurante",
    "roles": ["Freguês", "Garçom", "Cozinha"],
    "objects": ["Mesas", "Cadeiras", "Menu", "Refeição", "Dinheiro", "Gorjeta"],
    "entry_conditions": ["Freguês está com fome", "Freguês tem dinheiro"],
    "scenes": {
        "Cena 1: Entrar": [
            "Estacionar o carro",
            "Entrar no Restaurante",
            "Esperar por uma Mesa (ou Ir até a Mesa)",
            "Ler o Menu"
        ],
        "Cena 2: Pedir": [
            "Chamar Garçom",
            "Pedir Refeição",
            "Garçom leva pedido para Cozinha"
        ],
        "Cena 3: Comer": [
            "Cozinha prepara Refeição",
            "Garçom traz Refeição",
            "Freguês come Refeição"
        ],
        "Cena 4: Sair": [
            "Pedir a conta",
            "Pagar pela Refeição",
            "Dar Gorjeta (opcional)",
            "Sair do Restaurante"
        ]
    },
    "outcomes": ["Freguês não tem fome", "Freguês tem menos dinheiro", "Restaurante tem mais dinheiro"]
}

import json
print(json.dumps(restaurant_script, indent=2, ensure_ascii=False))

### Exercise: Build a Cinema Script

*(Based on Aula 03, slide 39)*

**Task:** Create a script for "Ir ao cinema" (Going to the cinema). Include a scene for buying popcorn and going to the restroom.

- **Papéis:** `Espectador`, `Atendente de Bilheteria`, `Atendente da Pipoca`
- **Objetos:** `Ingresso`, `Pipoca`, `Filme`, `Banheiro`
- **Condições de Entrada:** `Espectador quer ver o filme`, `Espectador tem dinheiro`
- **Cenas:** `Chegar ao cinema`, `Comprar ingresso`, `Comprar pipoca`, `Assistir ao filme` (this scene might be interrupted), `Ir ao banheiro`
- **Resultados:** `Espectador assistiu ao filme`, `Espectador tem menos dinheiro`


---

> Course materials adapted from slides by **Prof. Dr. Érica da Costa Reis Carvalho** (UFSJ)  
> Reference: **Russell, S.; Norvig, P. _Inteligência Artificial_, 2ª edição. Elsevier, 2004.**
